# App-8 : Modelisation declarative par contraintes (twin C# .NET)

> **Twin C# .NET de `App-8-MiniZinc.ipynb`** (marathon parite #4956). Kernel `.net-csharp`. Ce notebook utilise **Google.OrTools CP-SAT** (Prong A, EPIC #3801) -- le vrai moteur industriel de programmation par contraintes, accessible en .NET via NuGet. MiniZinc etant un langage de modelisation independant sans binding .NET officiel, ce twin exprime les memes modeles declaratifs via OR-Tools CP-SAT, qui partage le paradigme : **declarer les contraintes, laisser le solveur chercher**. Les graphiques matplotlib du twin Python sont remplaces par un rendu ASCII autonome.

## Objectifs d'apprentissage

1. **Comprendre** le paradigme declaratif (CP) face a l'imperatif (backtracking)
2. **Modeliser** en OR-Tools CP-SAT : variables, domaines, contraintes lineaires et globales
3. **Resolver** problemes classiques (equations, monnaie, N-Reines, emploi du temps, Sudoku)
4. **Comparer** la concision CP-SAT vs MiniZinc (du twin Python)
5. **Optimiser** : minimiser un objectif (monnaie, makespan, distance)

In [1]:
// App-8 : Modelisation declarative par contraintes -- twin C# de App-8-MiniZinc
// Prong A (#3801) : Google.OrTools CP-SAT, le vrai moteur industriel de programmation
// par contraintes. MiniZinc est un LANGAGE de modelisation independant (multi-solveur)
// sans binding .NET officiel ; ce twin exprime donc les memes modeles declaratifs via
// OR-Tools CP-SAT, qui partage le MEME paradigme (declarer les contraintes, laisser le
// solveur chercher). Le twin Python (App-8-MiniZinc.ipynb) utilise MiniZinc + OR-Tools.
#r "nuget: Google.OrTools"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using Google.OrTools.Sat;

// Culture invariante : separateur decimal "." (parite de sortie avec le twin Python,
// independant du locale FR de la machine hote). Les 3 setters sont necessaires car
// .NET Interactive evalue les cellules sur des threads du pool distincts.
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

Console.WriteLine("Environnement charge.");
Console.WriteLine("  .NET " + Environment.Version);
Console.WriteLine("  Google.OrTools CP-SAT importe (vrai moteur industriel, Prong A #3801)");


Installed Packages Google.OrTools, 9.15.6755

Environnement charge.


  .NET 10.0.9


  Google.OrTools CP-SAT importe (vrai moteur industriel, Prong A #3801)


In [2]:
// Utilitaire d'affichage d'un modele declare (numerotation des lignes), comme
// display_model() du twin Python. Permet de montrer la syntaxe MiniZinc cote-a-cote
// avec le code CP-SAT C# executable correspondant.
static void DisplayModel(string modelStr, string title = "Modele MiniZinc")
{
    string bar = new string('=', 60);
    Console.WriteLine("\n" + bar + "\n  " + title + "\n" + bar);
    var lines = modelStr.Trim().Split('\n');
    for (int i = 0; i < lines.Length; i++)
        Console.WriteLine($"  {i + 1,2} | {lines[i]}");
    Console.WriteLine(bar + "\n");
}

Console.WriteLine("Fonctions utilitaires definies.");


Fonctions utilitaires definies.


## 1. Introduction : pourquoi la programmation par contraintes ?

La **programmation par contraintes (CP)** renverse le paradigme imperatif : au lieu decrire *comment* resoudre (algorithme pas-a-pas), on decrit *quoi* resoudre (les contraintes que la solution doit satisfaire), et un **solveur** cherche. MiniZinc est un langage dedie (concis, multi-solveur) ; OR-Tools CP-SAT est un moteur industriel accessible via une API .NET. Ce twin utilise CP-SAT ; le twin Python montre MiniZinc cote-a-cote.

### Architecture

```
Modele (variables + contraintes)  -->  Solveur CP-SAT  -->  Solution
   (ce qu'on veut)                      (le moteur)        (ce qu'on obtient)
```

In [3]:
// === Section 2 : Premier modele -- systeme d'equations ===
// x + y = 10 ET x * y = 21 (x, y dans 1..10). Solution : x = 7, y = 3 (ou x = 3, y = 7).
//
// Equivalent MiniZinc (modele declare puis resolu par le twin Python) :
//   var 1..10: x;  var 1..10: y;
//   constraint x + y = 10;  constraint x * y = 21;
//   solve satisfy;

string mznEquations = @"
var 1..10: x;
var 1..10: y;
constraint x + y = 10;
constraint x * y = 21;
solve satisfy;";
DisplayModel(mznEquations, "Systeme d'equations (syntaxe MiniZinc)");

// Execution CP-SAT C# (Prong A : vrai moteur).
var model = new CpModel();
var x = model.NewIntVar(1, 10, "x");
var y = model.NewIntVar(1, 10, "y");
model.Add(x + y == 10);
var produit = model.NewIntVar(1, 100, "produit");
model.AddMultiplicationEquality(produit, new[] { x, y });
model.Add(produit == 21);

var solver = new CpSolver();
var status = solver.Solve(model);
Console.WriteLine($"Solution CP-SAT : x = {solver.Value(x)}, y = {solver.Value(y)}  (status: {status})");
Console.WriteLine($"Verification : x + y = {solver.Value(x) + solver.Value(y)}, x * y = {solver.Value(x) * solver.Value(y)}");



  Systeme d'equations (syntaxe MiniZinc)


   1 | var 1..10: x;


   2 | var 1..10: y;


   3 | constraint x + y = 10;


   4 | constraint x * y = 21;


   5 | solve satisfy;


Solution CP-SAT : x = 7, y = 3  (status: Optimal)


Verification : x + y = 10, x * y = 21


## 2. Syntaxe de base : premier modele

On commence par un **systeme d'equations** simple. MiniZinc (twin Python) l'exprime en 5 lignes ; CP-SAT C# ajoute la creation explicite des variables et le produit via `AddMultiplicationEquality`.

In [4]:
// === Section 2 (suite) : Optimisation -- probleme de monnaie ===
// Rendre 99 centimes avec le MINIMUM de pieces (1c, 5c, 10c, 25c).
// Solution optimale : 3x25c + 2x10c + 0x5c + 4x1c = 9 pieces.
//
// Equivalent MiniZinc : solve minimize total;  -->  CP-SAT : model.Minimize(total)

string mznCoins = @"
var 0..99: p1; var 0..19: p5; var 0..9: p10; var 0..3: p25;
constraint p1 + 5*p5 + 10*p10 + 25*p25 = 99;
var int: total = p1 + p5 + p10 + p25;
solve minimize total;";
DisplayModel(mznCoins, "Probleme de monnaie (optimisation, syntaxe MiniZinc)");

var mCoins = new CpModel();
var p1 = mCoins.NewIntVar(0, 99, "p1");
var p5 = mCoins.NewIntVar(0, 19, "p5");
var p10 = mCoins.NewIntVar(0, 9, "p10");
var p25 = mCoins.NewIntVar(0, 3, "p25");
mCoins.Add(p1 + 5 * p5 + 10 * p10 + 25 * p25 == 99);
var totalCoins = mCoins.NewIntVar(0, 200, "total");
mCoins.Add(totalCoins == p1 + p5 + p10 + p25);
mCoins.Minimize(totalCoins);   // <-- objectif : minimiser le nombre de pieces

var sCoins = new CpSolver();
sCoins.Solve(mCoins);
Console.WriteLine("Solution optimale CP-SAT :");
Console.WriteLine($"  {sCoins.Value(p25)}x25c + {sCoins.Value(p10)}x10c + {sCoins.Value(p5)}x5c + {sCoins.Value(p1)}x1c");
Console.WriteLine($"  Total = {sCoins.ObjectiveValue} pieces (minimum prouve par le solveur)");



  Probleme de monnaie (optimisation, syntaxe MiniZinc)


   1 | var 0..99: p1; var 0..19: p5; var 0..9: p10; var 0..3: p25;


   2 | constraint p1 + 5*p5 + 10*p10 + 25*p25 = 99;


   3 | var int: total = p1 + p5 + p10 + p25;


   4 | solve minimize total;


Solution optimale CP-SAT :


  3x25c + 2x10c + 0x5c + 4x1c


  Total = 9 pieces (minimum prouve par le solveur)


### Optimisation : probleme de monnaie

On passe de `solve satisfy` (trouver UNE solution) a `solve minimize` (trouver la MEILLEURE). En CP-SAT : `model.Minimize(objectif)`. Le solveur prouve l'optimalite (aucune solution a moins de pieces n'existe).

In [5]:
// === Section 3 : Contraintes globales -- N-Reines ===
// MiniZinc exprime N-Reines en ~5 lignes grace a la contrainte globale alldifferent.
// On presente la concision MiniZinc, puis l'equivalent CP-SAT C# executable.

string mznNQueens = @"
include ""globals.mzn"";
int: n = 8;
array[1..n] of var 1..n: q;
constraint alldifferent(q);                        % pas meme ligne
constraint alldifferent([q[i] + i | i in 1..n]);   % pas meme diagonale montante
constraint alldifferent([q[i] - i | i in 1..n]);   % pas meme diagonale descendante
solve satisfy;";
DisplayModel(mznNQueens, "N-Reines (syntaxe MiniZinc, ~5 lignes)");
Console.WriteLine("Comparaison de concision : MiniZinc ~5 lignes vs CP-SAT ~12 lignes (ratio ~2.4x).");



  N-Reines (syntaxe MiniZinc, ~5 lignes)


   1 | include "globals.mzn";


   2 | int: n = 8;


   3 | array[1..n] of var 1..n: q;


   4 | constraint alldifferent(q);                        % pas meme ligne


   5 | constraint alldifferent([q[i] + i | i in 1..n]);   % pas meme diagonale montante


   6 | constraint alldifferent([q[i] - i | i in 1..n]);   % pas meme diagonale descendante


   7 | solve satisfy;


Comparaison de concision : MiniZinc ~5 lignes vs CP-SAT ~12 lignes (ratio ~2.4x).


In [6]:
// N-Reines en CP-SAT C# (executable). q[i] = ligne de la reine en colonne i (0-indexe).
static (int[] solution, double ms) SolveNQueensCpsat(int n)
{
    var m = new CpModel();
    var q = new IntVar[n];
    for (int i = 0; i < n; i++) q[i] = m.NewIntVar(0, n - 1, $"q{i}");
    m.AddAllDifferent(q);   // pas deux reines sur la meme ligne
    // Diagonales : q[i]+i et q[i]-i toutes differentes
    var d1 = new IntVar[n]; var d2 = new IntVar[n];
    for (int i = 0; i < n; i++)
    {
        d1[i] = m.NewIntVar(0, 2 * n, $"d1{i}");
        d2[i] = m.NewIntVar(-n, n, $"d2{i}");
        m.Add(d1[i] == q[i] + i);
        m.Add(d2[i] == q[i] - i);
    }
    m.AddAllDifferent(d1);
    m.AddAllDifferent(d2);
    var s = new CpSolver();
    var sw = Stopwatch.StartNew();
    var status = s.Solve(m);
    sw.Stop();
    if (status == CpSolverStatus.Optimal || status == CpSolverStatus.Feasible)
    {
        var sol = new int[n];
        for (int i = 0; i < n; i++) sol[i] = (int)s.Value(q[i]);
        return (sol, sw.Elapsed.TotalMilliseconds);
    }
    return (null, sw.Elapsed.TotalMilliseconds);
}

var (sol8, ms8) = SolveNQueensCpsat(8);
Console.WriteLine($"CP-SAT solution 8-reines : [{string.Join(", ", sol8)}]");
Console.WriteLine($"CP-SAT temps             : {ms8:F2} ms");
Console.WriteLine("Echiquier (Q = reine, . = vide) :");
for (int row = 0; row < 8; row++)
{
    var line = "";
    for (int col = 0; col < 8; col++)
        line += (sol8[col] == row ? "Q " : ". ");
    Console.WriteLine("  " + line);
}


CP-SAT solution 8-reines : [7, 3, 0, 2, 5, 1, 6, 4]


CP-SAT temps             : 20.41 ms


Echiquier (Q = reine, . = vide) :


  . . Q . . . . . 


  . . . . . Q . . 


...

  . Q . . . . . . 


  . . . . . . . Q 


  . . . . Q . . . 


  . . . . . . Q . 


  Q . . . . . . . 


## 3. Contraintes globales

Les contraintes **globales** (alldifferent, cumulative, circuit) capturent des structures recurrentes. `alldifferent` est la plus utile : elle exprime en une ligne ce qui demanderait O(n^2) contraintes binaires.

In [7]:
// === Section 4 : Emploi du temps (timetabling) ===
// 6 cours, 5 creneaux, 3 salles, 3 enseignants. Contraintes C1/C2/C3.
// C1 : un enseignant ne donne pas 2 cours au meme creneau.
// C2 : une salle n'accueille qu'un cours par creneau (slot[i]!=slot[j] OU room[i]!=room[j]).
// C3 : la salle est assez grande pour les effectifs.
//
// C2 est une DISJONCTION (OU) -- en CP-SAT on la modelise avec des booleens et OnlyEnforceIf.

int nCourses = 6, nSlots = 5, nRooms = 3;
int[] teacher = { 1, 1, 2, 2, 3, 3 };
int[] students = { 30, 25, 40, 20, 35, 15 };
int[] roomCap = { 35, 45, 25 };

var mTt = new CpModel();
var slot = new IntVar[nCourses];
var room = new IntVar[nCourses];
for (int i = 0; i < nCourses; i++)
{
    slot[i] = mTt.NewIntVar(0, nSlots - 1, $"slot{i}");
    room[i] = mTt.NewIntVar(0, nRooms - 1, $"room{i}");
}

// C1 : enseignant ne donne pas 2 cours au meme creneau
for (int i = 0; i < nCourses; i++)
    for (int j = i + 1; j < nCourses; j++)
        if (teacher[i] == teacher[j])
            mTt.Add(slot[i] != slot[j]);

// C2 : une salle, un cours par creneau (disjonction reifiee)
for (int i = 0; i < nCourses; i++)
    for (int j = i + 1; j < nCourses; j++)
    {
        var bSlot = mTt.NewBoolVar($"sameSlot{i}_{j}");
        var bRoom = mTt.NewBoolVar($"sameRoom{i}_{j}");
        mTt.Add(slot[i] == slot[j]).OnlyEnforceIf(bSlot);
        mTt.Add(slot[i] != slot[j]).OnlyEnforceIf(bSlot.Not());
        mTt.Add(room[i] == room[j]).OnlyEnforceIf(bRoom);
        mTt.Add(room[i] != room[j]).OnlyEnforceIf(bRoom.Not());
        mTt.AddBoolOr(new[] { bSlot.Not(), bRoom.Not() });   // NON(meme slot ET meme room)
    }

// C3 : capacite de la salle suffisante
for (int i = 0; i < nCourses; i++)
    for (int r = 0; r < nRooms; r++)
        if (students[i] > roomCap[r])
            mTt.Add(room[i] != r);

var sTt = new CpSolver();
var stTt = sTt.Solve(mTt);
Console.WriteLine($"Emploi du temps -- status: {stTt}");
for (int i = 0; i < nCourses; i++)
    Console.WriteLine($"  Cours {i + 1} -> creneau {sTt.Value(slot[i]) + 1}, salle {sTt.Value(room[i]) + 1} (prof={teacher[i]}, {students[i]} eleves)");


Emploi du temps -- status: Optimal


  Cours 1 -> creneau 1, salle 1 (prof=1, 30 eleves)


  Cours 2 -> creneau 4, salle 3 (prof=1, 25 eleves)


  Cours 3 -> creneau 3, salle 2 (prof=2, 40 eleves)


  Cours 4 -> creneau 4, salle 2 (prof=2, 20 eleves)


  Cours 5 -> creneau 2, salle 1 (prof=3, 35 eleves)


  Cours 6 -> creneau 4, salle 1 (prof=3, 15 eleves)


### N-Reines en CP-SAT C#

Le solveur trouve une solution en quelques millisecondes. La concision est moindre qu'en MiniZinc (~12 vs ~5 lignes) car CP-SAT exige la creation explicite des variables de diagonale intermediaires.

In [8]:
// === Section 5 : Sudoku 4x4 pilote depuis le code ===
// Le twin Python pilote MiniZinc depuis Python (model + data .dzn).
// En CP-SAT C#, le modele et les donnees vivent dans le meme code : on fixe les
// cases initiales par des contraintes d'egalite, puis alldifferent sur lignes/colonnes/blocs.

int[,] initSudoku = { { 0, 0, 3, 0 }, { 3, 0, 0, 2 }, { 0, 3, 0, 0 }, { 0, 0, 2, 0 } };
Console.WriteLine("Grille initiale (0 = vide) :");
for (int i = 0; i < 4; i++)
    Console.WriteLine("  " + string.Join(" ", Enumerable.Range(0, 4).Select(j => initSudoku[i, j] > 0 ? initSudoku[i, j].ToString() : ".")));

var mSk = new CpModel();
var g = new IntVar[4, 4];
for (int i = 0; i < 4; i++)
    for (int j = 0; j < 4; j++)
    {
        g[i, j] = mSk.NewIntVar(1, 4, $"g{i}{j}");
        if (initSudoku[i, j] > 0) mSk.Add(g[i, j] == initSudoku[i, j]);
    }
for (int i = 0; i < 4; i++)
{
    mSk.AddAllDifferent(new[] { g[i, 0], g[i, 1], g[i, 2], g[i, 3] });   // ligne i
    mSk.AddAllDifferent(new[] { g[0, i], g[1, i], g[2, i], g[3, i] });   // colonne i
}
for (int br = 0; br < 2; br++)   // blocs 2x2
    for (int bc = 0; bc < 2; bc++)
        mSk.AddAllDifferent(new[] { g[br * 2, bc * 2], g[br * 2, bc * 2 + 1], g[br * 2 + 1, bc * 2], g[br * 2 + 1, bc * 2 + 1] });

var sSk = new CpSolver();
sSk.Solve(mSk);
Console.WriteLine("\nSolution CP-SAT :");
for (int i = 0; i < 4; i++)
    Console.WriteLine("  " + string.Join(" ", Enumerable.Range(0, 4).Select(j => sSk.Value(g[i, j]).ToString())));
Console.WriteLine("\nVoir la serie Sudoku pour des solveurs CSP complets -> ../../Sudoku/README.md");


Grille initiale (0 = vide) :


  . . 3 .


  3 . . 2


  . 3 . .


  . . 2 .


...

  4 2 3 1


  3 1 4 2


  2 3 1 4


  1 4 2 3



Voir la serie Sudoku pour des solveurs CSP complets -> ../../Sudoku/README.md


## 4. Emploi du temps (timetabling)

Un probleme reel : assigner creneaux et salles a 6 cours sans conflit (enseignant, salle, capacite). La contrainte C2 est une **disjonction** (OU) -- modelisee en CP-SAT avec des booleens et `OnlyEnforceIf` / `AddBoolOr`. C'est l'apport pedagogique cles : la logique disjonctive se traduit en variables booleennes de reification.

In [9]:
// === Section 6 : Comparaison imperatif vs declaratif ===
// N-Reines resolu de 2 facons : (1) backtracking pur (imperatif), (2) CP-SAT (declaratif).
// (MiniZinc, 3e approche du twin Python, est remplace par CP-SAT ici -- meme paradigme declaratif.)

static int[] SolveNQueensBacktrack(int n)
{
    int[] queens = new int[n];
    bool IsSafe(int col, int row)
    {
        for (int c = 0; c < col; c++)
            if (queens[c] == row || Math.Abs(c - col) == Math.Abs(queens[c] - row)) return false;
        return true;
    }
    bool Solve(int col)
    {
        if (col == n) return true;
        for (int row = 0; row < n; row++)
            if (IsSafe(col, row)) { queens[col] = row; if (Solve(col + 1)) return true; }
        return false;
    }
    return Solve(0) ? queens : null;
}

int nBench = 12;
var swBt = Stopwatch.StartNew();
var solBt = SolveNQueensBacktrack(nBench);
swBt.Stop();
Console.WriteLine($"Backtracking pur : {swBt.Elapsed.TotalMilliseconds,8:F2} ms  -> solution trouvee: {solBt != null}");

var (solCpsat12, msCpsat12) = SolveNQueensCpsat(nBench);
Console.WriteLine($"CP-SAT declaratif: {msCpsat12,8:F2} ms  -> solution trouvee: {solCpsat12 != null}");
Console.WriteLine($"\n{nBench}-Reines : les deux approches trouvent une solution valide.");


Backtracking pur :     0.43 ms  -> solution trouvee: True


CP-SAT declaratif:    20.11 ms  -> solution trouvee: True



12-Reines : les deux approches trouvent une solution valide.


In [10]:
// Tableau comparatif des 3 paradigmes (Python pur / CP-SAT / MiniZinc).
// Repris du twin Python, avec la colonne MiniZinc (high-level, multi-solveur).
var rows = new[]
{
    new { Crit = "Lignes de code (N-Reines)", Py = "~20 lignes", Cp = "~12 lignes", Mz = "~5 lignes" },
    new { Crit = "Lisibilite mathematique", Py = "Faible", Cp = "Moyenne", Mz = "Haute" },
    new { Crit = "Flexibilite du solveur", Py = "Aucune (ad-hoc)", Cp = "CP-SAT uniquement", Mz = "Multi-solveur" },
    new { Crit = "Integration Python/.NET", Py = "Totale", Cp = "Excellente (API native)", Mz = "Via package/CLI" },
    new { Crit = "Separation modele/donnees", Py = "Non", Cp = "Non (code)", Mz = "Oui (.mzn + .dzn)" },
    new { Crit = "Performance brute", Py = "Lente", Cp = "Tres rapide", Mz = "Depend du solveur" },
    new { Crit = "Cas d'usage ideal", Py = "Prototypage", Cp = "Production", Mz = "Modelisation, recherche" },
};
string sep = new string('-', 86);
Console.WriteLine("Comparaison des paradigmes de resolution\n" + new string('=', 86));
Console.WriteLine($"{"Critere",-30} {"Python pur",-18} {"CP-SAT",-18} {"MiniZinc",-18}");
Console.WriteLine(sep);
foreach (var r in rows)
    Console.WriteLine($"{r.Crit,-30} {r.Py,-18} {r.Cp,-18} {r.Mz,-18}");
Console.WriteLine(new string('=', 86));


Comparaison des paradigmes de resolution


Critere                        Python pur         CP-SAT             MiniZinc          


--------------------------------------------------------------------------------------


Lignes de code (N-Reines)      ~20 lignes         ~12 lignes         ~5 lignes         


Lisibilite mathematique        Faible             Moyenne            Haute             


Flexibilite du solveur         Aucune (ad-hoc)    CP-SAT uniquement  Multi-solveur     


Integration Python/.NET        Totale             Excellente (API native) Via package/CLI   


Separation modele/donnees      Non                Non (code)         Oui (.mzn + .dzn) 


Performance brute              Lente              Tres rapide        Depend du solveur 


Cas d'usage ideal              Prototypage        Production         Modelisation, recherche


## 5. Sudoku 4x4

Le twin Python pilote MiniZinc depuis Python (modele + donnees `.dzn`). En CP-SAT C#, le modele et les donnees vivent dans le meme code : on fixe les cases initiales par `Add(var == valeur)`, puis `AddAllDifferent` sur lignes, colonnes et blocs 2x2.

In [11]:
// Visualisation ASCII (remplace matplotlib) : concision et score multi-criteres.
static string Bar(int filled, int maxBars, char c = '#')
    => new string(c, filled) + new string(' ', maxBars - filled);

Console.WriteLine("Concision du modele (lignes de code estimees)\n" + new string('-', 50));
var problems = new[] { "N-Reines", "Emploi du temps", "Sudoku" };
var linesPy = new[] { 20, 60, 50 };
var linesCp = new[] { 12, 45, 35 };
var linesMz = new[] { 5, 25, 20 };
int maxL = 60;
for (int i = 0; i < 3; i++)
{
    Console.WriteLine($"\n{problems[i]}:");
    Console.WriteLine($"  Python pur {Bar(linesPy[i], maxL),-62} {linesPy[i]}");
    Console.WriteLine($"  CP-SAT     {Bar(linesCp[i], maxL),-62} {linesCp[i]}");
    Console.WriteLine($"  MiniZinc   {Bar(linesMz[i], maxL),-62} {linesMz[i]}");
}

Console.WriteLine("\n\nEvaluation multi-criteres (score 1-5)\n" + new string('-', 50));
var criteria = new[] { "Concision", "Lisibilite", "Portabilite", "Integr. .NET", "Performance" };
var scPy = new[] { 2, 2, 1, 5, 2 };
var scCp = new[] { 3, 3, 2, 5, 5 };
var scMz = new[] { 5, 5, 5, 3, 4 };
for (int i = 0; i < 5; i++)
{
    Console.WriteLine($"\n{criteria[i]}:");
    Console.WriteLine($"  Python pur {Bar(scPy[i], 5)}");
    Console.WriteLine($"  CP-SAT     {Bar(scCp[i], 5)}");
    Console.WriteLine($"  MiniZinc   {Bar(scMz[i], 5)}");
}


Concision du modele (lignes de code estimees)
--------------------------------------------------



N-Reines:


  Python pur ####################                                           20


  CP-SAT     ############                                                   12


  MiniZinc   #####                                                          5


...

  MiniZinc   ###  



Performance:


  Python pur ##   


  CP-SAT     #####


  MiniZinc   #### 


## 6. Comparaison : imperatif vs declaratif

On resout N-Reines des deux facons pour comparer. Le backtracking pur (imperatif) est plus lent et moins declaratif ; CP-SAT (declaratif) delegue au solveur. La troisieme approche du twin Python (MiniZinc) est omise ici (pas de binding .NET) -- CP-SAT la represente dans le paradigme declaratif.

### Tableau comparatif

In [12]:
// === Section 7 : Exercices (stubs C.1 -- a completer par l'etudiant) ===
// Exercice 1 : Carre magique 3x3. Les entiers 1..9 tous differents, chaque ligne/colonne/
// diagonale sommant a magic_sum = n*(n*n+1)/2 = 15 pour n=3. modele CP-SAT a completer.
static int[][] MagicSquareCpsat()   // TODO etudiant
{
    // Indice : 9 IntVar(1,9), AddAllDifferent sur les 9, puis Add(==15) sur chaque ligne,
    // colonne et les 2 diagonales. Retourner la grille 3x3 resolue, ou null.
    // Etape 1 : creer le modele et les 9 variables.
    // Etape 2 : ajouter AddAllDifferent et les contraintes de somme (magic_sum = 15).
    // Etape 3 : resoudre et extraire la grille.
    Console.WriteLine("Exercice a completer : carre magique 3x3 en CP-SAT");
    return null;
}
MagicSquareCpsat();


Exercice a completer : carre magique 3x3 en CP-SAT


In [13]:
// Exercice 2 : Coloration de graphe. Graphe A-B-C, A-D, B-E, C-F, D-E, E-F.
// Minimiser le nombre de couleurs (chromatique). modele CP-SAT a completer.
static int[] GraphColoringCpsat()   // TODO etudiant
{
    // Indice : 6 IntVar(0, maxColors-1) pour les sommets A..F. Pour chaque arete (u,v),
    // Add(color[u] != color[v]). Minimiser le nombre de couleurs distinctes (variable objectif).
    // Etape 1 : modeliser les contraintes d'aretes (voisins de couleur differente).
    // Etape 2 : chercher le minimum de couleurs (essayer k=1, 2, 3... jusqu'a faisabilite).
    Console.WriteLine("Exercice a completer : coloration de graphe en CP-SAT");
    return null;
}
GraphColoringCpsat();


Exercice a completer : coloration de graphe en CP-SAT


In [14]:
// Exercice 3 : VRP simplifie (6 noeuds, 1 depot). Minimiser la distance totale
// d'une tournee passant par tous les clients. modele CP-SAT a completer.
static int[] VrpCpsat()   // TODO etudiant
{
    // Indice : variables next[i] = successeur du noeud i. AddAllDifferent(next) pour une
    // permutation, contrainte de circuit (cycle hamiltonien), minimiser sum dist[i,next[i]].
    // Etape 1 : fixer les variables next et la permutation.
    // Etape 2 : ajouter l'elimination de sous-tours (ou utiliser AddCircuit si disponible).
    // Etape 3 : minimiser la distance totale.
    Console.WriteLine("Exercice a completer : VRP simplifie en CP-SAT");
    return null;
}
VrpCpsat();

Console.WriteLine("\n=== Recapitulatif ===");
Console.WriteLine("OR-Tools CP-SAT est le moteur declaratif CP de reference en .NET (Prong A #3801).");
Console.WriteLine("Il exprime les memes modeles que MiniZinc (alldifferent, sommes, optimisation)");
Console.WriteLine("avec une API imperative. MiniZinc reste superieur pour la concision et le");
Console.WriteLine("multi-solveur (Gecode, Chuffed, CBC...). Les deux partagent le paradigme :");
Console.WriteLine("declarer les contraintes, laisser le solveur chercher -- l'oppose du backtracking.");


Exercice a completer : VRP simplifie en CP-SAT



=== Recapitulatif ===


OR-Tools CP-SAT est le moteur declaratif CP de reference en .NET (Prong A #3801).


Il exprime les memes modeles que MiniZinc (alldifferent, sommes, optimisation)


avec une API imperative. MiniZinc reste superieur pour la concision et le


multi-solveur (Gecode, Chuffed, CBC...). Les deux partagent le paradigme :


declarer les contraintes, laisser le solveur chercher -- l'oppose du backtracking.


### Visualisation ASCII (concision et score multi-criteres)

Les graphiques matplotlib du twin Python sont remplaces par des barres ASCII autonomes. Deux axes : concision (lignes de code) et evaluation multi-criteres (score 1-5).